# LeetCode 1936 — Add Minimum Number of Rungs bằng Python

Link đề: https://leetcode.com/problems/add-minimum-number-of-rungs/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def addRungs(self, rungs: List[int], dist: int) -> int:
        ...
```

Lộ trình: **DP vét cạn trên mọi độ cao nguyên** (chỉ cho số nhỏ) → **mô phỏng leo + thêm từng nấc** `O(đáp án)` → **mỗi khoảng trống trừ dần `dist`** `O(Σ gap / dist)` → **công thức `(gap − 1) // dist` cho từng khoảng** `O(n)` (lời giải nộp).

## 1. Mục tiêu học

1. Tách bài toán thành các **khoảng trống độc lập** giữa hai nấc liên tiếp (kể cả từ mặt đất `0`).
2. Lập luận tham lam: trong mỗi khoảng, đặt nấc mới **xa nhất có thể** (cách `dist`) là tối ưu.
3. Rút ra công thức **số nấc cần thêm cho khoảng `g`** là `⌈g / dist⌉ − 1 = (g − 1) // dist` — và tránh lỗi lệch-1 khi `g` chia hết cho `dist`.
4. Viết DP vét cạn để kiểm chứng tham lam trên số nhỏ.
5. Mở rộng sang bài ngược: cho trước ngân sách nấc, tìm `dist` nhỏ nhất (tìm kiếm nhị phân).

## 2. Đề bài & ví dụ

Cho mảng số nguyên **tăng ngặt** `rungs` — độ cao các nấc thang. Bạn đang ở mặt đất (độ cao `0`) và muốn lên **nấc cuối cùng**.

Cho số nguyên `dist`. Bạn chỉ leo được lên **nấc cao kế tiếp** nếu khoảng cách từ chỗ đang đứng (mặt đất hoặc một nấc) tới nấc đó **không quá** `dist`. Bạn được **thêm nấc** ở bất kỳ độ cao **nguyên dương** nào chưa có nấc.

Trả về số nấc **ít nhất** cần thêm để leo được tới nấc cuối.

**Ví dụ 1:** `rungs = [1,3,5,10], dist = 2` → `2` (thêm ở 7 và 8; thang thành `[1,3,5,7,8,10]`).

**Ví dụ 2:** `rungs = [3,6,8,10], dist = 3` → `0`.

**Ví dụ 3:** `rungs = [3,4,6,7], dist = 2` → `1` (không lên được nấc đầu từ mặt đất; thêm ở 1).

**Ràng buộc:** `1 ≤ rungs.length ≤ 10⁵`, `1 ≤ rungs[i] ≤ 10⁹`, `1 ≤ dist ≤ 10⁹`, `rungs` tăng ngặt.

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [([1, 3, 5, 10], 2, 2), ([3, 6, 8, 10], 3, 0), ([3, 4, 6, 7], 2, 1)]
for r, d, want in examples:
    print(r, "dist =", d, "-> kỳ vọng", want)

[1, 3, 5, 10] dist = 2 -> kỳ vọng 2
[3, 6, 8, 10] dist = 3 -> kỳ vọng 0
[3, 4, 6, 7] dist = 2 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Khoảng trống giữa các nấc (gồm mặt đất)

Thêm `0` vào đầu: các khoảng là `rungs[0] − 0, rungs[1] − rungs[0], …`. `zip` với bản dịch giúp viết gọn.

In [2]:
rungs = [1, 3, 5, 10]
heights = [0] + rungs
gaps = [b - a for a, b in zip(heights, heights[1:])]
print("các khoảng:", gaps)

các khoảng: [1, 2, 2, 5]


### 3.2 Làm tròn lên bằng số nguyên

`⌈g / d⌉ = (g + d − 1) // d`; và `⌈g / d⌉ − 1 = (g − 1) // d` với `g ≥ 1`.

In [3]:
d = 2
for g in range(1, 8):
    print(f"g={g}: ceil(g/d)-1 = {(g + d - 1) // d - 1} | (g-1)//d = {(g - 1) // d}")

g=1: ceil(g/d)-1 = 0 | (g-1)//d = 0
g=2: ceil(g/d)-1 = 0 | (g-1)//d = 0
g=3: ceil(g/d)-1 = 1 | (g-1)//d = 1
g=4: ceil(g/d)-1 = 1 | (g-1)//d = 1
g=5: ceil(g/d)-1 = 2 | (g-1)//d = 2
g=6: ceil(g/d)-1 = 2 | (g-1)//d = 2
g=7: ceil(g/d)-1 = 3 | (g-1)//d = 3


### 3.3 Chia một đoạn thành các bước ≤ `d`

Đi quãng `g` với mỗi bước tối đa `d` cần ít nhất `⌈g / d⌉` **bước**, tức `⌈g / d⌉ − 1` **điểm dừng ở giữa** (nấc mới).

In [4]:
g, d = 5, 2
stops = list(range(d, g, d))
print(f"quãng {g}, bước {d}: dừng tại {stops} -> {len(stops)} nấc thêm, {len(stops) + 1} bước")

quãng 5, bước 2: dừng tại [2, 4] -> 2 nấc thêm, 3 bước


### 3.4 Số nguyên lớn

`g` tới `10⁹`, `dist` có thể là 1 ⇒ đáp án tới ~`10⁹`. Không được thêm **từng nấc một** trong vòng lặp.

In [5]:
print("ví dụ cực đoan:", (10**9 - 1) // 1, "nấc thêm cho một khoảng")

ví dụ cực đoan: 999999999 nấc thêm cho một khoảng


### 3.5 Vì sao "đặt xa nhất" không bao giờ thua?

So sánh hai cách đặt nấc trong khoảng `0 → 10` với `dist = 3`: tham lam `3, 6, 9` (3 nấc) với một cách "rụt rè" `2, 4, 6, 8` (4 nấc). Mỗi lần tham lam luôn đứng **cao hơn hoặc bằng** cách kia sau cùng số nấc — nên không bao giờ cần nhiều nấc hơn. Đây là lập luận "luôn đi trước" (greedy stays ahead).

In [6]:
def heights_after(stops):
    return list(zip(range(1, len(stops) + 1), stops))

greedy = [3, 6, 9]; timid = [2, 4, 6, 8]
print("tham lam:", heights_after(greedy))
print("rụt rè  :", heights_after(timid))
print("sau k nấc, tham lam luôn cao hơn hoặc bằng:", all(g >= t for g, t in zip(greedy, timid)))

tham lam: [(1, 3), (2, 6), (3, 9)]
rụt rè  : [(1, 2), (2, 4), (3, 6), (4, 8)]
sau k nấc, tham lam luôn cao hơn hoặc bằng: True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `rungs = [1,3,5,10]`, `dist = 2`:

| đoạn | khoảng `g` | `g ≤ 2`? | cần thêm |
|---|---|---|---|
| 0 → 1 | 1 | ✔ | 0 |
| 1 → 3 | 2 | ✔ | 0 |
| 3 → 5 | 2 | ✔ | 0 |
| 5 → 10 | 5 | ✘ | thêm 7, 9 (hoặc 7, 8): **2** |

Tổng 2. Quan sát: các đoạn **độc lập** (một nấc thêm chỉ nằm trong một đoạn), và trong đoạn dài `g` cần chia thành các bước `≤ dist` ⇒ `⌈g/dist⌉ − 1` nấc.

Ví dụ 3: đoạn `0 → 3` dài 3 > 2 ⇒ cần `⌈3/2⌉ − 1 = 1` nấc (đặt ở 1 hoặc 2).

In [7]:
for r, d, want in examples:
    h = [0] + r
    print(r, d, [(b - a, (b - a - 1) // d) for a, b in zip(h, h[1:])], "-> tổng", sum((b - a - 1) // d for a, b in zip(h, h[1:])))

[1, 3, 5, 10] 2 [(1, 0), (2, 0), (2, 0), (5, 2)] -> tổng 2
[3, 6, 8, 10] 3 [(3, 0), (3, 0), (2, 0), (2, 0)] -> tổng 0
[3, 4, 6, 7] 2 [(3, 1), (1, 0), (2, 0), (1, 0)] -> tổng 1


## 5. Cách 1 — Brute Force / baseline: DP trên mọi độ cao nguyên

**Ý tưởng:** không giả định gì về tham lam. `best[h]` = số nấc thêm ít nhất để **đứng** ở độ cao `h` (với `h` là nấc có sẵn hoặc nấc thêm), quy tắc: từ `h` lên được `h'` nếu `0 < h' − h ≤ dist` **và không có nấc có sẵn nào bị bỏ qua** (phải leo lên nấc **kế tiếp** — không được nhảy qua nấc có sẵn). Chỉ dùng được khi độ cao nhỏ.

In [8]:
def add_rungs_brute(rungs, dist):
    top = rungs[-1]
    original = set(rungs)
    INF = float("inf")
    best = [INF] * (top + 1)
    best[0] = 0
    nxt_orig = [0] * (top + 2)                    # nxt_orig[h] = nấc có sẵn nhỏ nhất > h
    nxt = top + 1
    for h in range(top, -1, -1):
        nxt_orig[h] = nxt
        if h in original:
            nxt = h
    for h in range(top + 1):
        if best[h] == INF:
            continue
        for h2 in range(h + 1, min(h + dist, nxt_orig[h], top) + 1):   # không vượt qua nấc có sẵn kế tiếp
            cost = best[h] + (0 if h2 in original else 1)
            if cost < best[h2]:
                best[h2] = cost
    return best[top]

for r, d, want in examples:
    print(add_rungs_brute(r, d), "| kỳ vọng", want)

2 | kỳ vọng 2
0 | kỳ vọng 0
1 | kỳ vọng 1


**Độ phức tạp:** `O(top · dist)` — với `top = 10⁹` bất khả. **Nút thắt:** ta đang xét từng độ cao; thực ra mỗi đoạn giữa hai nấc có sẵn là một bài con độc lập có nghiệm dạng đóng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (độc lập):** vì phải leo qua **mọi** nấc có sẵn theo thứ tự, các nấc thêm nằm trong các đoạn `(h_{i−1}, h_i)` (với `h₋₁ = 0`) và mỗi đoạn được giải riêng.

**Insight 2 (đoạn đơn):** đoạn dài `g` cần số bước tối thiểu `⌈g / dist⌉` (mỗi bước ≤ `dist`), tức số nấc thêm `⌈g/dist⌉ − 1 = (g − 1) // dist`. Đặt tham lam ở `h + dist, h + 2·dist, …` đạt được con số này.

**Quy tắc:** `ans = Σ_i (h_i − h_{i−1} − 1) // dist`.

**Invariant khi duyệt:** sau khi xử lý nấc `i`, `ans` là số nấc thêm tối thiểu để đứng được ở nấc `i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng leo, thêm từng nấc khi bị kẹt (`O(đáp án + n)`)

In [9]:
def add_rungs_simulate(rungs, dist):
    cur, added = 0, 0
    for r in rungs:
        while r - cur > dist:          # không với tới: đặt nấc mới xa nhất có thể
            cur += dist
            added += 1
        cur = r
    return added

for r, d, want in examples:
    print(add_rungs_simulate(r, d), want)

2 2
0 0
1 1


**Độ phức tạp:** mỗi nấc thêm một vòng `while` ⇒ `O(n + đáp án)`. Với `rungs = [10⁹]`, `dist = 1`: `10⁹` vòng (đo ở mục 11). Nút thắt: thêm **từng** nấc ⇒ tính số lượng **một phát** bằng phép chia.

### 7.2 Cách 3 — Công thức theo từng khoảng (`O(n)`) — lời giải nộp

In [10]:
def add_rungs_formula_demo(rungs, dist):
    prev, ans = 0, 0
    for r in rungs:
        g = r - prev
        need = (g - 1) // dist
        print(f"{prev:>3} -> {r:>3}: khoảng {g}, cần thêm (g-1)//dist = {need}")
        ans += need
        prev = r
    return ans

print("=>", add_rungs_formula_demo([1, 3, 5, 10], 2))

  0 ->   1: khoảng 1, cần thêm (g-1)//dist = 0
  1 ->   3: khoảng 2, cần thêm (g-1)//dist = 0
  3 ->   5: khoảng 2, cần thêm (g-1)//dist = 0
  5 ->  10: khoảng 5, cần thêm (g-1)//dist = 2
=> 2


In [11]:
class SolutionFormula:
    def addRungs(self, rungs: List[int], dist: int) -> int:
        prev = ans = 0
        for r in rungs:
            ans += (r - prev - 1) // dist
            prev = r
        return ans

for r, d, want in examples:
    print(SolutionFormula().addRungs(r, d), want)

2 2
0 0
1 1


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Viết gọn bằng `zip` và `sum`

In [12]:
class SolutionZip:
    def addRungs(self, rungs: List[int], dist: int) -> int:
        return sum((b - a - 1) // dist for a, b in zip([0] + rungs, rungs))

for r, d, want in examples:
    print(SolutionZip().addRungs(r, d), want)

2 2
0 0
1 1


`[0] + rungs` tạo list mới `O(n)` — chấp nhận được. Có thể dùng `itertools.chain([0], rungs)` để tránh sao chép nếu muốn.

### 7.4 Cách 5 — Vector hóa bằng NumPy

Khi dữ liệu lớn và đã ở dạng mảng số, có thể tính tất cả khoảng cùng lúc: `np.diff` trên `[0] + rungs`, rồi `(gaps − 1) // dist` và cộng. Cùng công thức, chạy bằng mã C.

In [13]:
import numpy as np

class SolutionNumpy:
    def addRungs(self, rungs: List[int], dist: int) -> int:
        h = np.concatenate(([0], np.asarray(rungs, dtype=np.int64)))
        return int(((np.diff(h) - 1) // dist).sum())

for r, d, want in examples:
    print(SolutionNumpy().addRungs(r, d), want)

2 2
0 0
1 1


**Độ phức tạp:** `O(n)` (hằng số nhỏ hơn vòng lặp Python khi `n` lớn, nhưng tốn chi phí chuyển đổi list → mảng). Dùng `int64` để tổng không tràn.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 3 `rungs = [3,4,6,7]`, `dist = 2`:

| `prev` | `r` | `g = r − prev` | `(g − 1) // 2` | `ans` |
|---|---|---|---|---|
| 0 | 3 | 3 | 1 | 1 |
| 3 | 4 | 1 | 0 | 1 |
| 4 | 6 | 2 | 0 | 1 |
| 6 | 7 | 1 | 0 | **1** |

Và trace Cách 2 để thấy **vị trí** nấc thêm (tham lam đặt xa nhất):

In [14]:
def trace_simulate(rungs, dist):
    cur, added, placed = 0, 0, []
    for r in rungs:
        while r - cur > dist:
            cur += dist; added += 1; placed.append(cur)
            print(f"  kẹt trước nấc {r}: thêm nấc tại {cur}")
        print(f"leo lên nấc {r} (từ {cur}, cách {r - cur})")
        cur = r
    print("nấc thêm:", placed, "-> tổng", added)
    return added

trace_simulate([3, 4, 6, 7], 2); print(); trace_simulate([1, 3, 5, 10], 2)

  kẹt trước nấc 3: thêm nấc tại 2
leo lên nấc 3 (từ 2, cách 1)
leo lên nấc 4 (từ 3, cách 1)
leo lên nấc 6 (từ 4, cách 2)
leo lên nấc 7 (từ 6, cách 1)
nấc thêm: [2] -> tổng 1

leo lên nấc 1 (từ 0, cách 1)
leo lên nấc 3 (từ 1, cách 2)
leo lên nấc 5 (từ 3, cách 2)
  kẹt trước nấc 10: thêm nấc tại 7
  kẹt trước nấc 10: thêm nấc tại 9
leo lên nấc 10 (từ 9, cách 1)
nấc thêm: [7, 9] -> tổng 2


2

## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def addRungs(self, rungs: List[int], dist: int) -> int:
        ans = 0
        prev = 0                                   # bắt đầu từ mặt đất
        for r in rungs:
            ans += (r - prev - 1) // dist          # = ceil(gap / dist) - 1
            prev = r
        return ans

for r, d, want in examples:
    assert Solution().addRungs(r, d) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Phân rã.** Vì chỉ được leo lên nấc **kế tiếp** cao hơn, người leo đi qua mọi nấc có sẵn `h₀ < h₁ < … ` theo thứ tự. Các nấc thêm trong đoạn `(h_{i−1}, h_i)` chỉ ảnh hưởng việc vượt đoạn đó ⇒ tổng tối thiểu = tổng tối thiểu từng đoạn.

**Một đoạn dài `g`.** *Cận dưới:* nếu thêm `k` nấc thì đoạn được chia thành `k + 1` bước, mỗi bước `≤ dist` ⇒ `g ≤ (k + 1)·dist` ⇒ `k ≥ ⌈g/dist⌉ − 1`. *Đạt được:* đặt nấc ở `h_{i−1} + dist, h_{i−1} + 2dist, …, h_{i−1} + k·dist` với `k = ⌈g/dist⌉ − 1`; khoảng cuối `g − k·dist ≤ dist` ✔, mọi độ cao là số nguyên dương và nằm trong đoạn (không trùng nấc có sẵn).

**Công thức.** `⌈g/d⌉ − 1 = (g − 1) // d` với `g ≥ 1`: viết `g − 1 = q·d + r`, `0 ≤ r < d` ⇒ `g = q·d + (r + 1)` với `1 ≤ r + 1 ≤ d` ⇒ `⌈g/d⌉ = q + 1`.

Kiểm chứng bằng máy hai mệnh đề: (1) `⌈g/d⌉ − 1 == (g − 1) // d` với mọi `g, d ≥ 1` trong một miền lớn; (2) cận dưới — không có cách đặt nào dùng **ít hơn** `(g−1)//d` nấc: thử mọi tập nấc có kích thước nhỏ hơn trên khoảng nhỏ.

In [16]:
from itertools import combinations as _comb
assert all(-(-g // d) - 1 == (g - 1) // d for g in range(1, 3000) for d in range(1, 60))
for g in range(1, 13):
    for d in range(1, 6):
        need = (g - 1) // d
        if need == 0:
            continue
        for stops in _comb(range(1, g), need - 1):         # thử mọi tập ít hơn 1 nấc
            pts = [0, *stops, g]
            assert any(b - a > d for a, b in zip(pts, pts[1:]))
print("công thức đúng và cận dưới chặt trên mọi khoảng g ≤ 12, d ≤ 5")

công thức đúng và cận dưới chặt trên mọi khoảng g ≤ 12, d ≤ 5


## 11. Phân tích Time / Space

Gọi `n = len(rungs) ≤ 10⁵`, `A` = đáp án (≤ ~`10⁹`).

| Cách | Time | Space |
|---|---|---|
| 1. DP mọi độ cao | `O(top · dist)` | `O(top)` |
| 2. Mô phỏng thêm từng nấc | `O(n + A)` | `O(1)` |
| 3. Công thức | `O(n)` | `O(1)` |
| 4. `zip` + `sum` | `O(n)` | `O(n)` (list `[0] + rungs`) |

In [17]:
rng = random.Random(1936)
big = sorted(rng.sample(range(1, 10**9), 10**5))
for name, sol in [("Công thức/Nộp", Solution()), ("zip", SolutionZip()), ("NumPy", SolutionNumpy())]:
    t = time.perf_counter(); r = sol.addRungs(big, 7)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")
t = time.perf_counter(); r = add_rungs_simulate([2 * 10**6], 1); dt = time.perf_counter() - t
print(f"mô phỏng với A = 2·10^6: {dt:.2f}s -> A = 10^9 ≈ {dt * 500:.0f}s | công thức: {Solution().addRungs([10**9], 1)}")

Công thức/Nộp: 142799695 trong 5.3 ms
zip: 142799695 trong 10.9 ms
NumPy: 142799695 trong 3.0 ms


mô phỏng với A = 2·10^6: 0.07s -> A = 10^9 ≈ 35s | công thức: 999999999


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu DP)

In [18]:
sols = [Solution(), SolutionFormula(), SolutionZip(), SolutionNumpy()]
edge = examples + [
    ([1], 1, 0),
    ([5], 1, 4),
    ([4], 2, 1),             # g chia hết cho dist: chỉ cần 1 nấc (ở 2)
    ([6], 3, 1),
    ([7], 3, 2),
    ([1, 2, 3], 1, 0),
    ([10, 20, 30], 10, 0),
    ([10, 20, 31], 10, 1),
]
for r, d, want in edge:
    assert add_rungs_brute(r, d) == want, (r, d, add_rungs_brute(r, d))
    for s in sols:
        assert s.addRungs(r, d) == want, (r, d, s)
    assert add_rungs_simulate(r, d) == want
    print("OK", r, "dist =", d, "->", want)

OK [1, 3, 5, 10] dist = 2 -> 2
OK [3, 6, 8, 10] dist = 3 -> 0
OK [3, 4, 6, 7] dist = 2 -> 1
OK [1] dist = 1 -> 0
OK [5] dist = 1 -> 4
OK [4] dist = 2 -> 1
OK [6] dist = 3 -> 1
OK [7] dist = 3 -> 2
OK [1, 2, 3] dist = 1 -> 0
OK [10, 20, 30] dist = 10 -> 0
OK [10, 20, 31] dist = 10 -> 1


### 12.2 Ngẫu nhiên có seed: so với DP vét cạn

In [19]:
rng = random.Random(42)
for _ in range(600):
    n = rng.randint(1, 6)
    rungs = sorted(rng.sample(range(1, 40), n))
    d = rng.randint(1, 8)
    want = add_rungs_brute(rungs, d)
    for s in sols:
        assert s.addRungs(rungs, d) == want, (rungs, d)
    assert add_rungs_simulate(rungs, d) == want
for _ in range(100):
    rungs = sorted(rng.sample(range(1, 10**9), rng.randint(1, 500)))
    d = rng.randint(1, 10**6)
    assert len({s.addRungs(rungs, d) for s in sols} | {add_rungs_simulate(rungs, d)}) == 1
print("600 case nhỏ (DP) + 100 case lớn (mô phỏng): khớp")

600 case nhỏ (DP) + 100 case lớn (mô phỏng): khớp


### 12.3 Kiểm tra thang sau khi thêm thật sự leo được

In [20]:
def placed_rungs(rungs, dist):
    cur, out = 0, []
    for r in rungs:
        while r - cur > dist:
            cur += dist; out.append(cur)
        cur = r
    return out

rng = random.Random(7)
for _ in range(300):
    rungs = sorted(rng.sample(range(1, 200), rng.randint(1, 10))); d = rng.randint(1, 15)
    extra = placed_rungs(rungs, d)
    full = sorted(set(rungs) | set(extra))
    assert len(full) == len(rungs) + len(extra)               # không trùng nấc có sẵn
    assert all(b - a <= d for a, b in zip([0] + full, full))  # leo được
    assert len(extra) == Solution().addRungs(rungs, d)
print("thang sau khi thêm luôn leo được, số nấc thêm đúng bằng đáp án")

thang sau khi thêm luôn leo được, số nấc thêm đúng bằng đáp án


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `g // dist` (thừa 1 khi chia hết)

In [21]:
def wrong_floor(rungs, dist):
    prev = ans = 0
    for r in rungs:
        ans += (r - prev) // dist if r - prev > dist else 0      # SAI khi g chia hết dist
        prev = r
    return ans

for r, d in [([4], 2), ([1, 3, 5, 10], 2), ([6], 3)]:
    print(r, d, "sai:", wrong_floor(r, d), "| đúng:", Solution().addRungs(r, d))

[4] 2 sai: 2 | đúng: 1
[1, 3, 5, 10] 2 sai: 2 | đúng: 2
[6] 3 sai: 2 | đúng: 1


**Vì sao sai:** khoảng `4` với `dist = 2` chỉ cần **1** nấc (ở độ cao 2); `4 // 2 = 2` đếm thừa. Số nấc thêm = số **bước** − 1 = `⌈g/d⌉ − 1`. Với `g` không chia hết (như 5) thì `g // d = ⌈g/d⌉ − 1` tình cờ đúng — nên Ví dụ 1 không bắt được lỗi.

### Lỗi 2 — Quên khoảng từ mặt đất lên nấc đầu

In [22]:
def wrong_no_floor(rungs, dist):
    return sum((b - a - 1) // dist for a, b in zip(rungs, rungs[1:]))   # SAI: bỏ đoạn 0 -> rungs[0]

for r, d, want in examples:
    print(r, d, "sai:", wrong_no_floor(r, d), "| đúng:", want)

[1, 3, 5, 10] 2 sai: 2 | đúng: 2
[3, 6, 8, 10] 3 sai: 0 | đúng: 0
[3, 4, 6, 7] 2 sai: 0 | đúng: 1


**Vì sao sai:** người leo bắt đầu ở **mặt đất** (độ cao 0) — Ví dụ 3 cần thêm nấc ngay ở đoạn đầu. **Sửa:** `prev = 0` ban đầu (hoặc `[0] + rungs`).

### Lỗi 3 — Thêm từng nấc trong vòng lặp (`while`) với `dist` nhỏ

In [23]:
t = time.perf_counter(); r1 = add_rungs_simulate([3 * 10**6], 1); dt = time.perf_counter() - t
print(f"rungs = [3·10^6], dist = 1: {r1} nấc, mất {dt:.2f}s; với 10^9 sẽ ≈ {dt * 333:.0f}s (TLE)")

rungs = [3·10^6], dist = 1: 2999999 nấc, mất 0.10s; với 10^9 sẽ ≈ 34s (TLE)


**Vì sao sai:** đúng về kết quả nhưng đáp án có thể tới `10⁹` ⇒ số vòng lặp tới `10⁹`. **Sửa:** tính số nấc mỗi khoảng bằng phép chia.

### Lỗi 4 — Dùng `math.ceil(g / dist) − 1` với số thực

In [24]:
import math
g, d = 10**17 + 1, 10**17                  # ngoài ràng buộc, minh họa sai số float
print("float:", math.ceil(g / d) - 1, "| nguyên:", (g - 1) // d)
print("trong ràng buộc thì vẫn khớp:", all(math.ceil(g / d) - 1 == (g - 1) // d
      for g in range(1, 2000) for d in range(1, 50)))

float: 0 | nguyên: 1
trong ràng buộc thì vẫn khớp: True


**Vì sao nguy hiểm:** với ràng buộc `≤ 10⁹`, số thực double vẫn đủ chính xác, nhưng `g / d` là phép chia số thực; với số lớn hơn `2⁵³` có thể làm tròn sai. Thói quen tốt: `(g − 1) // d`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi bước tối đa `d`, chèn ít điểm nhất để phủ các khoảng"** ⇒ mỗi khoảng `g` cần `⌈g/d⌉ − 1` điểm chèn.
- Các khoảng giữa các điểm **bắt buộc** đi qua là **độc lập** ⇒ cộng lại.
- Nhớ điểm xuất phát (mặt đất, gốc tọa độ…) là một "nấc" ẩn.
- Tránh vòng lặp theo **giá trị** (có thể tới `10⁹`) — dùng phép chia.
- Bài ngược "cho ngân sách, tìm `d` nhỏ nhất" ⇒ tìm kiếm nhị phân trên `d` (Bài tập 5) — giống LeetCode 1870, 2064.

## 15. Biến thể của bài

1. **Không bắt buộc đi qua nấc có sẵn** (được nhảy qua) ⇒ vẫn như cũ vì nấc có sẵn không bao giờ làm hại (đứng lên nó miễn phí).
2. **Tìm `dist` nhỏ nhất** khi được thêm tối đa `k` nấc ⇒ nhị phân trên `dist` (Bài tập 5).
3. **Chi phí thêm nấc khác nhau theo độ cao** ⇒ DP trên các đoạn.
4. **LeetCode 1870 — Minimum Speed to Arrive on Time**, **2064** — cùng mẫu "tìm tham số nhỏ nhất bằng nhị phân + hàm đếm ⌈·⌉".
5. **Trạm xăng / LeetCode 871** — chèn trạm có sẵn tối thiểu (heap) — biến thể tham lam khó hơn.

Minh họa một biến thể (tự dựng): nấc thêm chỉ được đặt ở độ cao là **bội của `t`**. Tham lam "đặt xa nhất có thể" vẫn đúng: từ độ cao `cur`, chọn bội lớn nhất của `t` không vượt `min(cur + dist, nấc kế tiếp)`; nếu không có bội nào cao hơn `cur` thì không leo được (`-1`). Đối chiếu với DP vét cạn.

In [25]:
def add_rungs_multiple(rungs, dist, t):
    cur, added = 0, 0
    for r in rungs:
        while r - cur > dist:
            nxt = (cur + dist) // t * t
            if nxt <= cur:
                return -1
            cur = nxt; added += 1
        cur = r
    return added

def add_rungs_multiple_brute(rungs, dist, t):
    top = rungs[-1]; original = set(rungs); INF = float("inf")
    best = [INF] * (top + 1); best[0] = 0
    nxt_o = [0] * (top + 2); nx = top + 1
    for h in range(top, -1, -1):
        nxt_o[h] = nx
        if h in original: nx = h
    for h in range(top + 1):
        if best[h] == INF: continue
        for h2 in range(h + 1, min(h + dist, nxt_o[h], top) + 1):
            if h2 in original:
                best[h2] = min(best[h2], best[h])
            elif h2 % t == 0:
                best[h2] = min(best[h2], best[h] + 1)
    return -1 if best[top] == INF else best[top]

rng_m = random.Random(17)
for _ in range(500):
    rungs = sorted(rng_m.sample(range(1, 50), rng_m.randint(1, 5)))
    d = rng_m.randint(1, 8); t = rng_m.randint(1, 5)
    assert add_rungs_multiple(rungs, d, t) == add_rungs_multiple_brute(rungs, d, t), (rungs, d, t)
print("biến thể 'bội của t': tham lam khớp DP; ví dụ:", add_rungs_multiple([1, 3, 5, 10], 2, 2), add_rungs_multiple([7], 3, 4))

biến thể 'bội của t': tham lam khớp DP; ví dụ: 2 -1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP mọi độ cao | vét cạn có thứ tự | `O(top·dist)` | `O(top)` | kiểm chứng số nhỏ |
| 2. Mô phỏng | thêm nấc xa nhất mỗi lần kẹt | `O(n + A)` | `O(1)` | cần **vị trí** nấc thêm, `A` nhỏ |
| 3. Công thức | `(g − 1) // dist` | `O(n)` | `O(1)` | **nộp bài** |
| 4. `zip` + `sum` | như 3, một dòng | `O(n)` | `O(n)` | viết gọn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `([5], 1)`, `([4], 2)`, `([2, 8], 3)`?

**Bài 2 (tính tay):** Lập bảng khoảng và số nấc thêm cho `rungs = [4, 8, 9, 20]`, `dist = 4`.

**Bài 3 (điền TODO):** Viết `added_positions(rungs, dist)` trả về **danh sách độ cao** các nấc thêm (theo tham lam) mà không lặp từng nấc khi danh sách rất dài? (Gợi ý: dùng `range` với bước `dist`.)

**Bài 4 (tự viết lại):** Viết `can_climb(rungs, dist, k)` — với tối đa `k` nấc thêm có leo được không — chỉ một dòng `sum(...) <= k`.

**Bài 5 (mini challenge):** Cho `rungs` và ngân sách `k`, tìm `dist` **nhỏ nhất** để leo được với tối đa `k` nấc thêm.

In [26]:
def added_positions(rungs, dist):
    # TODO
    pass

def can_climb(rungs, dist, k):
    # TODO
    pass

def min_dist_with_budget(rungs, k):
    # TODO: nhị phân trên dist
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `(g−1)//d` | `[5],1` → 4; `[4],2` → 1 | `[2,8],3`: `(2−1)//3 + (6−1)//3 = 0 + 1` |
| 2 | Khoảng `4, 4, 1, 11` | `(3//4, 3//4, 0, 10//4)` | Tổng `0 + 0 + 0 + 2 = 2` |
| 3 | Mỗi khoảng `(prev, r)`: `range(prev + dist, r, dist)` | Nhưng phải dừng khi còn cách `r` ≤ `dist` | Số phần tử đúng `(g−1)//dist` |
| 4 | Tổng số nấc cần ≤ `k` | Dùng công thức | `sum((b-a-1)//dist …) <= k` |
| 5 | `cần(dist)` không tăng theo `dist` | Nhị phân `dist ∈ [1, max gap]` | Tìm `dist` nhỏ nhất có `cần(dist) ≤ k` |

## 19. Đáp án tham khảo

In [27]:
r1 = [Solution().addRungs(*a) for a in [([5], 1), ([4], 2), ([2, 8], 3)]]
print("Bài 1:", r1); assert r1 == [4, 1, 1]
print("Bài 2:"); assert add_rungs_formula_demo([4, 8, 9, 20], 4) == 2

Bài 1: [4, 1, 1]
Bài 2:
  0 ->   4: khoảng 4, cần thêm (g-1)//dist = 0
  4 ->   8: khoảng 4, cần thêm (g-1)//dist = 0
  8 ->   9: khoảng 1, cần thêm (g-1)//dist = 0
  9 ->  20: khoảng 11, cần thêm (g-1)//dist = 2


In [28]:
def added_positions(rungs, dist):
    out, prev = [], 0
    for r in rungs:
        k = (r - prev - 1) // dist
        out.extend(range(prev + dist, prev + dist * k + 1, dist))   # đúng k nấc
        prev = r
    return out

def can_climb(rungs, dist, k):
    return sum((b - a - 1) // dist for a, b in zip([0] + rungs, rungs)) <= k

rng = random.Random(3)
for _ in range(400):
    rungs = sorted(rng.sample(range(1, 300), rng.randint(1, 12))); d = rng.randint(1, 20)
    assert added_positions(rungs, d) == placed_rungs(rungs, d)
    k = rng.randint(0, 30)
    assert can_climb(rungs, d, k) == (add_rungs_brute(rungs, d) <= k)
print("Bài 3, 4: OK", added_positions([1, 3, 5, 10], 2))

Bài 3, 4: OK [7, 9]


In [29]:
def min_dist_with_budget(rungs, k):
    gaps = [b - a for a, b in zip([0] + rungs, rungs)]
    lo, hi = 1, max(gaps)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((g - 1) // mid for g in gaps) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

rng = random.Random(9)
for _ in range(400):
    rungs = sorted(rng.sample(range(1, 120), rng.randint(1, 8))); k = rng.randint(0, 10)
    brute = next(d for d in range(1, 200) if Solution().addRungs(rungs, d) <= k)
    assert min_dist_with_budget(rungs, k) == brute, (rungs, k)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", min_dist_with_budget([1, 3, 5, 10], 0), min_dist_with_budget([1, 3, 5, 10], 1))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 5 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans, prev = 0, 0                       # bắt đầu từ mặt đất
for r in rungs:
    ans += (r - prev - 1) // dist      # ceil(gap/dist) - 1
    prev = r
return ans
```

**Checklist:**
- [ ] Tính cả khoảng **từ mặt đất** (`prev = 0`)?
- [ ] Công thức `(g − 1) // dist` (không phải `g // dist`)?
- [ ] Không thêm từng nấc trong vòng lặp?
- [ ] Không dùng số thực / `math.ceil`?
- [ ] `O(n)`.